# 🤖 Capstone C1 · Build a flow-matching "action expert" that pushes a T

**Capstone · stage 1 of 4** &nbsp;|&nbsp; ⏱ 60–90 min &nbsp;|&nbsp; 🖥️ Colab **T4 GPU** recommended (*Runtime → Change runtime type*). CPU works but is slower.

### What you're building
A robot policy that looks at the scene and outputs the **next 16 moves** (an *action chunk*), generated with **flow matching**: noise → actions along straight lines, exactly like lab 07's moons.

### Why this matters for industry
| This notebook | The same idea at scale |
|---|---|
| Flow-matching velocity net over action chunks | **Physical Intelligence π0 / π0.5** "action expert", **NVIDIA GR00T N1** diffusion-transformer action head, Hugging Face **SmolVLA** |
| Action chunking + receding horizon | ACT (ALOHA), Diffusion Policy (Toyota Research / Columbia), every modern VLA |
| Fixed-seed simulator evaluation with confidence intervals | how robotics teams at Google DeepMind and Toyota Research report policy results |

### What you'll have at the end
✅ a trained policy · ✅ its score on 50 fixed test scenes, with a 95% interval · ✅ GIFs of successes and failures · ✅ an experiment showing *why generative policies* · ✅ a checkpoint for stages C3 and C4.

**Pace:** 🧩 challenges are small blanks (the key idea only). If you skip one, the notebook explains it and uses a working answer.

In [ ]:
#@title 🔧 Step 0 · Run this cell first (click ▶). It loads the tools for this lab. { display-mode: "form" }
import importlib.util, subprocess, sys, os
def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
if importlib.util.find_spec("gym_pusht") is None:
    _pip("gym-pusht==0.1.6", "pymunk>=6.6,<7")          # pymunk 7 removed an API gym-pusht needs
if importlib.util.find_spec("zarr") is None or not __import__("zarr").__version__.startswith("2."):
    _pip("zarr==2.18.7", "numcodecs==0.15.1")
if importlib.util.find_spec("imageio") is None:
    _pip("imageio")

import numpy as np, torch, math

# ---------------------------------------------------------------------------
# Guided-lab helpers. You never need to edit this cell.
#  * ___            : a blank for you to fill in
#  * check(name, x) : checks your answer; if it is blank or wrong, it explains
#                     and hands back a working version so the notebook keeps going
#  * quiz(id)       : a clickable multiple-choice question
#  * playground(...) : sliders that re-run a function when you let go
# ---------------------------------------------------------------------------
import inspect, html as _html
import numpy as np
from IPython.display import display, HTML
import os
try:
    import ipywidgets as widgets
    _WIDGETS = not os.environ.get("GUIDE_NO_WIDGETS")
except Exception:
    _WIDGETS = False

class BlankNotFilled(Exception):
    pass

class _Blank:
    """The ___ placeholder. Any maths with it stops with a friendly message."""
    __array_ufunc__ = None
    def _stop(self, *args, **kwargs):
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = _stop
    __truediv__ = __rtruediv__ = __floordiv__ = __rfloordiv__ = _stop
    __pow__ = __rpow__ = __matmul__ = __rmatmul__ = __mod__ = __rmod__ = _stop
    __neg__ = __pos__ = __abs__ = __getitem__ = __call__ = __iter__ = _stop
    __lt__ = __le__ = __gt__ = __ge__ = __bool__ = __float__ = __int__ = __index__ = _stop
    __array__ = __len__ = _stop
    def __getattr__(self, name):
        if name.startswith('__'):
            raise AttributeError(name)
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    def __repr__(self):
        return "___"

___ = _Blank()
CHALLENGES, QUIZZES = {}, {}
_solved, _quiz_score = {}, {}

_STYLE = {
    "ok":   ("#e8f6ee", "#1b7a4b", "✅"),
    "wait": ("#fff5e0", "#9a5b00", "🧩"),
    "bad":  ("#fdecea", "#b3261e", "❌"),
    "info": ("#eaf1fb", "#245eb5", "💡"),
}

def card(kind, title, body=""):
    bg, fg, icon = _STYLE[kind]
    display(HTML(
        f'<div style="background:{bg};border-left:5px solid {fg};padding:10px 14px;'
        f'border-radius:6px;margin:6px 0;color:#1d2530;font-size:14px;line-height:1.5">'
        f'<b style="color:{fg}">{icon} {title}</b><div>{body}</div></div>'))

def _as_numpy(x):
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    if isinstance(x, (list, tuple)):
        return [_as_numpy(v) for v in x]
    return x

def _same(a, b, tol):
    a, b = _as_numpy(a), _as_numpy(b)
    if isinstance(a, list) or isinstance(b, list):
        return isinstance(a, list) and isinstance(b, list) and len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    try:
        a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    except Exception:
        return a == b
    return a.shape == b.shape and np.allclose(a, b, atol=tol, rtol=tol)

def _has_blank(obj):
    if isinstance(obj, _Blank):
        return True
    if isinstance(obj, dict):
        return any(_has_blank(v) for v in obj.values())
    if isinstance(obj, (list, tuple)):
        return any(_has_blank(v) for v in obj)
    if callable(obj):
        try:
            return "___" in inspect.getsource(obj)
        except Exception:
            return False
    return False

def check(name, answer):
    """Check a challenge. Returns your answer if it works, otherwise a working reference."""
    ch = CHALLENGES[name]
    ref = ch["reference"]
    title = ch.get("title", name)
    fallback = ("<br><i>For now the notebook will use a working version so every later cell still runs. "
                "Come back, fill it in, and re-run this cell.</i>")
    if _has_blank(answer):
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” is waiting for you", "Hint: " + ch["hint"] + fallback)
        return ref
    try:
        if "test" in ch:
            ok, message = ch["test"](answer)
        elif callable(ref):
            ok, message = True, ""
            for args in ch["cases"]:
                args = args if isinstance(args, tuple) else (args,)
                expected, got = ref(*args), answer(*args)
                if not _same(expected, got, ch.get("tol", 1e-6)):
                    ok = False
                    message = "For a test input your function gave a different result from the expected one."
                    break
        else:
            ok = _same(ref, answer, ch.get("tol", 1e-6))
            message = f"You entered <code>{_html.escape(repr(_as_numpy(answer)))}</code>."
    except BlankNotFilled:
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” still has a blank", "Hint: " + ch["hint"] + fallback)
        return ref
    except Exception as err:
        ok, message = False, f"Running your version raised <code>{_html.escape(type(err).__name__)}: {_html.escape(str(err))}</code>."
    if ok:
        _solved[name] = True
        card("ok", f"Challenge solved: {title}", ch.get("why", ""))
        return answer
    _solved[name] = False
    card("bad", f"Not quite yet: {title}", message + "<br>Hint: " + ch["hint"] + fallback)
    return ref

def quiz(qid):
    q = QUIZZES[qid]
    question = f'<div style="font-size:15px;margin:8px 0 4px"><b>{"🔮 Predict: " if q.get("predict") else "🤔 "}{q["q"]}</b></div>'
    if not _WIDGETS:
        options = "".join(f"<li>{_html.escape(o)}</li>" for o in q["options"])
        display(HTML(question + f"<ol type='A'>{options}</ol><details><summary>Answer</summary>"
                     f"{'ABCDEFG'[q['answer']]}. {q['explain']}</details>"))
        return
    out = widgets.Output()
    buttons = []
    def choose(i):
        def handler(_):
            _quiz_score.setdefault(qid, i == q["answer"])
            for j, b in enumerate(buttons):
                b.button_style = "success" if j == q["answer"] else ("danger" if j == i else "")
            with out:
                out.clear_output()
                if i == q["answer"]:
                    card("ok", "Yes!", q["explain"])
                else:
                    card("bad", "Not this one. Here is the reasoning:", q["explain"])
        return handler
    for i, option in enumerate(q["options"]):
        b = widgets.Button(description=f"{'ABCDEFG'[i]}. {option}", layout=widgets.Layout(width="auto", max_width="100%"))
        b.on_click(choose(i))
        buttons.append(b)
    display(HTML(question), widgets.VBox(buttons), out)

def playground(fn, **controls):
    """controls: name=(min, max, step, default) for sliders, or name=[option, ...] for a dropdown."""
    defaults, sliders = {}, {}
    for name, spec in controls.items():
        if isinstance(spec, list):
            defaults[name] = spec[0]
            if _WIDGETS:
                sliders[name] = widgets.Dropdown(options=spec, value=spec[0], description=name)
        else:
            lo, hi, step, value = spec
            defaults[name] = value
            if _WIDGETS:
                kind = widgets.IntSlider if all(isinstance(v, int) for v in spec) else widgets.FloatSlider
                sliders[name] = kind(min=lo, max=hi, step=step, value=value, description=name,
                                     continuous_update=False, style={"description_width": "initial"},
                                     layout=widgets.Layout(width="420px"))
    if _WIDGETS:
        ui = widgets.VBox(list(sliders.values()))
        out = widgets.interactive_output(fn, sliders)
        display(ui, out)
    else:
        fn(**defaults)

def progress_report():
    solved = sum(_solved.values()); total = len(CHALLENGES)
    right = sum(_quiz_score.values()); asked = len(_quiz_score)
    stars = "⭐" * solved + "☆" * (total - solved)
    body = f"Challenges solved yourself: <b>{solved} / {total}</b> {stars}<br>"
    body += f"Quiz questions right on the first click: <b>{right} / {asked}</b> (of {len(QUIZZES)} in this lab)"
    missing = [CHALLENGES[k].get('title', k) for k in CHALLENGES if not _solved.get(k)]
    if missing:
        body += "<br>Still worth a try: " + ", ".join(missing)
    card("info", "Your progress in this lab", body)

# ---- this lab's challenges and quizzes ----
CHALLENGES["score"] = dict(title="The PushT score",
    reference=lambda best_coverage: min(best_coverage / 0.95, 1.0),
    cases=[0.5, 0.95, 0.99, 0.0],
    hint="Divide the best coverage by 0.95 (the success line), then cap it at 1.0 with <code>min(…, 1.0)</code>.",
    why="This is the metric from the Diffusion Policy paper. It rewards partial progress and treats anything above 95% coverage as a full success.")

CHALLENGES["to_unit"] = dict(title="Normalise to −1…1", reference=lambda x: x / 256.0 - 1.0,
    cases=[np.array([0.0, 256.0, 512.0])],
    hint="0 should map to −1, 256 to 0, 512 to +1: divide by 256, then subtract 1.",
    why="Flow matching starts from standard Gaussian noise, so targets should live on a similar scale (about −1…1). Unnormalised targets are a classic cause of a policy that never learns.")

CHALLENGES["chunk_ids"] = dict(title="Which future actions form the chunk?",
    reference=lambda i, episode_end, H: np.minimum(np.arange(i, i + H), episode_end - 1),
    cases=[(10, 100, 16), (95, 100, 16), (0, 3, 5)],
    hint="Take indices i, i+1, …, i+H−1, but never go past the episode's last frame (<code>episode_end − 1</code>): <code>np.minimum(np.arange(i, i + H), episode_end - 1)</code>.",
    why="Near the end of a demonstration we repeat the final action, which means “hold still”. Leaking into the <i>next</i> episode would teach nonsense moves.")

def _ref_fm_pieces(x0, x1, t):
    return (1 - t) * x0 + t * x1, x1 - x0
def _test_fm(fn):
    x0, x1, t = torch.randn(4, 16, 2), torch.randn(4, 16, 2), torch.rand(4, 1, 1)
    got, ref = fn(x0, x1, t), _ref_fm_pieces(x0, x1, t)
    ok = all(torch.allclose(g, r) for g, r in zip(got, ref))
    return ok, "Check both lines: the noisy point on the straight line, and the velocity along it."
CHALLENGES["fm_pieces"] = dict(title="Flow matching for action chunks", reference=_ref_fm_pieces, test=_test_fm,
    hint="Same two lines as lab 07: <code>(1 - t) * x0 + t * x1</code> and <code>x1 - x0</code>. <code>t</code> has shape (B, 1, 1), so it broadcasts over the 16×2 chunk.",
    why="That's the entire training signal of π0's action expert: regress the straight-line velocity from noise to the demonstrated action chunk.")

def _test_euler(fn):
    const = lambda x, t, obs: torch.ones_like(x)
    x = fn(const, torch.zeros(3, 12), torch.zeros(3, 16, 2), 5)
    ok = torch.allclose(x, torch.ones(3, 16, 2))
    return ok, "After integrating a constant velocity of 1 from t=0 to t=1, every number should have moved by exactly 1."
def _ref_euler(velocity_net, obs, x, flow_steps):
    dt = 1.0 / flow_steps
    for k in range(flow_steps):
        t = torch.full((len(x), 1), k * dt, device=x.device)
        x = x + dt * velocity_net(x, t, obs)
    return x
CHALLENGES["euler"] = dict(title="Integrate noise into actions", reference=_ref_euler, test=_test_euler,
    hint="One Euler step: <code>x + dt * velocity_net(x, t, obs)</code>.",
    why="Ten tiny steps turn random noise into a smooth, plausible 1.6-second motion plan in a few milliseconds.")

CHALLENGES["execute"] = dict(title="Receding horizon", reference=lambda chunk, execute: chunk[:execute],
    cases=[(np.arange(32).reshape(16, 2), 8), (np.arange(32).reshape(16, 2), 16)],
    hint="Only the first <code>execute</code> actions of the chunk are carried out before re-planning: <code>chunk[:execute]</code>.",
    why="Committing to part of a plan keeps motion smooth (no dithering between strategies). Re-planning keeps it reactive (lab 01's MPC).")

QUIZZES["metric"] = dict(predict=True, q="Human demonstrations in this dataset reach at most about 85–90% coverage. What does that imply for the “success = coverage > 95%” metric?",
    options=["Our policy should easily reach 100% success", "Full success will be rare even for good policies, so we also need a partial-credit score", "The simulator must be broken"],
    answer=1, explain="We measured every demo: none crosses 95%. That's why Diffusion Policy reports the <b>score</b> (best coverage ÷ 0.95, capped at 1) and why we compare against the human score on the same scale.")
QUIZZES["chunk"] = dict(q="Why predict a whole <b>chunk</b> of 16 future actions instead of just the next one?",
    options=["It is 16× faster to train", "Chunks capture consistent multi-step motions and reduce jittery switching between strategies", "The simulator requires 16 actions at a time"],
    answer=1, explain="Humans move in strokes, not isolated twitches. A chunk commits to one coherent stroke. ACT, Diffusion Policy, π0 and GR00T all predict chunks.")
QUIZZES["mse"] = dict(predict=True, q="A <b>regression</b> policy (same network, trained with plain MSE to output the chunk directly) will score…",
    options=["about the same or better: its training loss gets much lower", "lower on average: when demonstrations disagree (around the T's left or right side) it averages them", "exactly zero"],
    answer=1, explain="MSE rewards predicting the <i>average</i> of the demonstrated futures. When two strategies are valid, their average can be invalid, such as driving straight into the T. In our test run regression scored 0.75 vs 0.87 for flow matching (paired difference +0.12, 95% CI +0.01 to +0.23) and collapsed (score below 0.3) in 11 of 50 scenes versus 3, even though its training loss was 100× lower. Compare with your own numbers below.")
QUIZZES["steps"] = dict(predict=True, q="Generating each chunk with only <b>1</b> flow step instead of 10 will make the policy…",
    options=["much worse", "about the same", "better"],
    answer=0, explain="In our test run 1 step dropped the score from about 0.87 to about 0.3. Like lab 07's one-step blob, it outputs an <i>average</i> of possible motions. But 5 steps already matched 10, so flow matching needs only a handful of network calls per plan, fast enough for real-time robot control.")
print('✅ Setup complete. Scroll down and run the cells in order.')

In [ ]:
#@title 🧰 Capstone toolkit · run me (data, simulator, evaluation, GIFs). Read the notes below; no need to edit. { display-mode: "form" }
import os, json, time, math, copy, hashlib, urllib.request
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
import gymnasium as gym, gym_pusht, zarr, imageio
from IPython.display import Image as _GifImage
plt.rcParams.update({"figure.dpi": 110})

FAST_DEV_RUN = os.environ.get("CAPSTONE_FAST") == "1"      # course authors' quick self-test switch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
OUT = Path("capstone_outputs"); OUT.mkdir(exist_ok=True)
print("device:", DEVICE, "· outputs folder:", OUT.resolve())
if DEVICE.type == "cpu":
    print("⚠️  No GPU found. In Colab: Runtime → Change runtime type → T4 GPU. (CPU works, just slower.)")

# ---------------- data ----------------
DATA_URL = "https://diffusion-policy.cs.columbia.edu/data/training/pusht.zip"
def load_pusht(with_images=False):
    archive = Path("pusht.zip")
    if not archive.exists():
        print("Downloading PushT (31 MB)…"); urllib.request.urlretrieve(DATA_URL, archive)
    root = zarr.open_group(store=zarr.ZipStore(str(archive), mode="r"), mode="r", path="pusht/pusht_cchi_v7_replay.zarr")
    data = dict(state=root["data/state"][:].astype(np.float32), action=root["data/action"][:].astype(np.float32),
                ends=root["meta/episode_ends"][:])
    data["starts"] = np.r_[0, data["ends"][:-1]]
    if with_images:
        imgs = np.empty((len(data["state"]), 96, 96, 3), np.uint8)
        for i in range(0, len(imgs), 2048):
            imgs[i:i + 2048] = root["data/img"][i:i + 2048]
        data["images"] = imgs
    order = np.random.default_rng(42).permutation(len(data["ends"]))         # the SAME split as lab 04
    data["train_eps"], data["val_eps"], data["test_eps"] = order[:144], order[144:175], order[175:]
    return data

# ---------------- simulator ----------------
def make_env(obs_type="state"):
    return gym.make("gym_pusht/PushT-v0", obs_type=obs_type)

def sim_state(env):
    u = env.unwrapped
    return np.array([*u.agent.position, *u.block.position, u.block.angle % (2 * np.pi)], np.float32)

def set_dataset_state(env, s):
    """Put the simulator exactly into a recorded dataset state.
    gym-pusht's own reset_to_state sets the block angle AFTER its position. With modern pymunk that rotates
    the T around its centre of mass and moves it ~90 units away from where the dataset recorded it.
    Setting the angle FIRST reproduces the recorded frames pixel-for-pixel (we verified this on all 206 demos)."""
    u = env.unwrapped
    u.agent.position = list(map(float, s[:2])); u.agent.velocity = (0, 0)
    u.block.angle = float(s[4]); u.block.position = list(map(float, s[2:4]))
    u.block.velocity = (0, 0); u.block.angular_velocity = 0
    u.space.step(u.dt)
    return u.get_obs()

def coverage_of(env):
    return float(env.unwrapped._get_coverage())

def score_from_best_coverage(best):
    return min(best / 0.95, 1.0)

# ---------------- evaluation ----------------
EVAL_SEEDS = list(range(100000, 100050))       # 50 fixed start states, identical for every experiment

def run_episode(choose_chunk, seed, execute=8, obs_type="state", record=False, max_steps=300):
    """choose_chunk(history) -> array of future actions (world units). history = list of past observations."""
    env = make_env(obs_type)
    obs, _ = env.reset(seed=seed)
    history, best, frames, steps = [obs, obs], 0.0, [], 0
    while steps < max_steps:
        chunk = choose_chunk(history[-2:], env)
        for a in chunk[:execute]:
            obs, _, terminated, _, info = env.step(np.asarray(a, np.float64))
            history.append(obs); steps += 1
            best = max(best, info["coverage"])
            if record and steps % 2 == 0:
                frames.append(env.unwrapped.render()[::3, ::3])
            if terminated or steps >= max_steps:
                break
        if terminated:
            break
    env.close()
    return dict(best_coverage=best, score=score_from_best_coverage(best), success=best > 0.95, steps=steps, frames=frames)

def bootstrap_ci(values, repeats=2000, seed=0):
    values = np.asarray(values, float); r = np.random.default_rng(seed)
    means = values[r.integers(0, len(values), (repeats, len(values)))].mean(1)
    return float(values.mean()), float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975))

def evaluate(choose_chunk, seeds=EVAL_SEEDS, execute=8, obs_type="state", label="policy", verbose=True):
    t0 = time.time(); rows = [run_episode(choose_chunk, s, execute, obs_type) for s in seeds]
    scores = [r["score"] for r in rows]
    mean, lo, hi = bootstrap_ci(scores)
    result = dict(label=label, score=mean, ci95=[lo, hi], success_rate=float(np.mean([r["success"] for r in rows])),
                  reached_80pct=float(np.mean([r["best_coverage"] > 0.8 for r in rows])), episodes=len(rows),
                  seconds=round(time.time() - t0, 1), per_episode_scores=scores)
    if verbose:
        print(f"{label}: score {mean:.3f} (95% CI {lo:.3f}–{hi:.3f}) · ≥80% coverage in {result['reached_80pct']:.0%} "
              f"· full success {result['success_rate']:.0%} · {len(rows)} episodes in {result['seconds']:.0f}s")
    return result

def save_gif(frames, name):
    path = OUT / name
    imageio.mimsave(path, frames, duration=0.1, loop=0)
    return path

def show_gif(path, width=280):
    display(_GifImage(filename=str(path), width=width))

def save_results(name, obj):
    (OUT / name).write_text(json.dumps(obj, indent=1))
    print("saved", OUT / name)

def human_reference(data, episodes):
    env = make_env("state"); env.reset(seed=0); best = []
    for e in episodes:
        b = 0.0
        for i in range(data["starts"][e], data["ends"][e]):
            set_dataset_state(env, data["state"][i]); b = max(b, coverage_of(env))
        best.append(score_from_best_coverage(b))
    return float(np.mean(best))


def to_unit(x):          # world units 0..512  ->  -1..1
    return x / 256.0 - 1.0
def from_unit(x):        # -1..1  ->  world units 0..512
    return (x + 1.0) * 256.0
def state_features(s):   # (…, 5) -> (…, 6): positions in -1..1, angle as sin & cos
    return np.concatenate([to_unit(s[..., :4]), np.sin(s[..., 4:5]), np.cos(s[..., 4:5])], -1).astype(np.float32)

class ResBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.norm = nn.LayerNorm(width)
        self.ff = nn.Sequential(nn.Linear(width, 4 * width), nn.SiLU(), nn.Linear(4 * width, width))
    def forward(self, h):
        return h + self.ff(self.norm(h))

class FlowPolicy(nn.Module):
    """Velocity network v(noisy action chunk, time, observation)."""
    def __init__(self, obs_dim=12, horizon=16, act_dim=2, width=512, depth=3):
        super().__init__()
        self.horizon, self.act_dim = horizon, act_dim
        self.inp = nn.Linear(horizon * act_dim + obs_dim + 32, width)
        self.blocks = nn.Sequential(*[ResBlock(width) for _ in range(depth)])
        self.out = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, horizon * act_dim))
        self.register_buffer("freqs", torch.exp(torch.linspace(0, math.log(1000), 16)))
    def forward(self, x_t, t, obs):
        tf = t * self.freqs
        h = self.inp(torch.cat([x_t.flatten(1), obs, torch.sin(tf), torch.cos(tf)], 1))
        return self.out(self.blocks(h)).view(-1, self.horizon, self.act_dim)

---
## 1 · Meet the task 🎮
The dataset is lab 04's 206 human demonstrations. The simulator is **gym-pusht**, the same physics used to record them.

> 🐛 **A real bug we hit while building this capstone.** gym-pusht's built-in *"reset to this recorded state"* placed the T ~90 units away from where the dataset recorded it (a pymunk version change). Replaying demonstrations silently failed. The toolkit's `set_dataset_state` fixes it: replayed demos now match the recorded images pixel for pixel. **Lesson:** always check that your simulator reproduces your data before trusting any evaluation.

In [ ]:
data = load_pusht()
S, A_, starts, ends = data["state"], data["action"], data["starts"], data["ends"]
print(f"{len(ends)} demos · train {len(data['train_eps'])} / val {len(data['val_eps'])} / test {len(data['test_eps'])}")

env = make_env("state"); env.reset(seed=0)
first = data["test_eps"][0]
set_dataset_state(env, S[starts[first]])
fig, axs = plt.subplots(1, 2, figsize=(7, 3.5))
axs[0].imshow(env.unwrapped.render()); axs[0].set_title("start of a test demo"); axs[0].axis("off")
set_dataset_state(env, S[ends[first] - 1])
axs[1].imshow(env.unwrapped.render()); axs[1].set_title(f"its end · coverage {coverage_of(env):.0%}"); axs[1].axis("off")
plt.show()

### 🎛️ Playground · Push it yourself
Start from a recorded scene. Choose where the pusher should go (`target_x`, `target_y`) and for how many steps. Can you raise the coverage?

In [ ]:
def push_by_hand(target_x=250, target_y=300, steps=20, demo=0):
    e = data["test_eps"][demo]
    env = make_env("state"); env.reset(seed=0); set_dataset_state(env, S[starts[e]])
    before = coverage_of(env)
    for _ in range(steps):
        env.step(np.array([target_x, target_y], float))
    img = env.unwrapped.render()
    plt.figure(figsize=(3.6, 3.6)); plt.imshow(img)
    k = img.shape[0] / 512; plt.plot(target_x * k, target_y * k, "rx", ms=12, mew=3)
    plt.title(f"coverage {before:.0%} → {coverage_of(env):.0%}"); plt.axis("off"); plt.show()

playground(push_by_hand, target_x=(0, 512, 8, 250), target_y=(0, 512, 8, 300), steps=(1, 60, 1, 20), demo=(0, 30, 1, 0))

In [ ]:
quiz("metric")

### 🧩 Challenge 1 · The PushT score

In [ ]:
def score_from_best_coverage(best_coverage):
    return ___     # 🧩 partial credit, capped at a full success

score_from_best_coverage = check("score", score_from_best_coverage)
HUMAN = human_reference(data, data["test_eps"])
print(f"Human demonstrators on the 31 test episodes score {HUMAN:.3f}. That's our reference line.")

<details><summary>🤔 <b>Need a hint?</b></summary>

Scale by the success threshold, cap at 1.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return min(best_coverage / 0.95, 1.0)     # 🧩 partial credit, capped at a full success</pre>

</details>

---
## 2 · Turn demonstrations into training examples 🧱

Each training example is a pair:
* **observation:** the last **2** states (pusher x, y, block x, y, block angle), so the policy can sense motion (lab 02).
* **target:** the next **16** actions (1.6 seconds of motion).

In [ ]:
quiz("chunk")

### 🧩 Challenge 2 · Normalise to −1…1

In [ ]:
def to_unit(x):
    return ___        # 🧩 0 → −1, 256 → 0, 512 → +1

to_unit = check("to_unit", to_unit)

<details><summary>🤔 <b>Need a hint?</b></summary>

Divide by half the world size, then shift.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return x / 256.0 - 1.0        # 🧩 0 → −1, 256 → 0, 512 → +1</pre>

</details>

### 🧩 Challenge 3 · Which future actions form the chunk?

In [ ]:
H, EXECUTE = 16, 8          # predict 16 actions, carry out 8, then re-plan

def chunk_indices(i, episode_end, H):
    return ___   # 🧩 i … i+H−1, never past the episode's end

chunk_indices = check("chunk_ids", chunk_indices)

obs_list, chunk_list = [], []
for e in data["train_eps"]:
    for i in range(starts[e], ends[e]):
        prev = max(i - 1, starts[e])                                   # at the first frame, repeat it
        obs_list.append(np.concatenate([state_features(S[prev]), state_features(S[i])]))
        chunk_list.append(to_unit(A_[chunk_indices(i, ends[e], H)]))
OBS = torch.tensor(np.array(obs_list)).to(DEVICE)          # (examples, 12)
CHUNKS = torch.tensor(np.array(chunk_list)).to(DEVICE)     # (examples, 16, 2)
print("observations", tuple(OBS.shape), "· action chunks", tuple(CHUNKS.shape))

e, i = data["train_eps"][3], starts[data["train_eps"][3]] + 40
env = make_env("state"); env.reset(seed=0); set_dataset_state(env, S[i]); img = env.unwrapped.render(); k = img.shape[0] / 512
plt.figure(figsize=(3.6, 3.6)); plt.imshow(img)
c = A_[chunk_indices(i, ends[e], H)] * k
plt.plot(c[:, 0], c[:, 1], "r.-"); plt.plot(c[0, 0], c[0, 1], "go"); plt.title("one target chunk (green = first move)"); plt.axis("off"); plt.show()

<details><summary>🤔 <b>Need a hint?</b></summary>

<code>np.arange</code> for the range, <code>np.minimum</code> to clip.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return np.minimum(np.arange(i, i + H), episode_end - 1)   # 🧩 i … i+H−1, never past the episode's end</pre>

</details>

---
## 3 · The action expert network 🧠
A small residual MLP. Input: the noisy chunk (32 numbers), the observation (12), and the flow time `t` described by sine waves (32). Output: a velocity for every number in the chunk. It's defined in the toolkit cell as `FlowPolicy`, and it has about **6.4 M** parameters.

π0's action expert is a 300 M-parameter transformer doing the same job, conditioned on a vision-language model instead of 12 numbers.

### 🧩 Challenge 4 · Flow matching for action chunks

In [ ]:
def flow_matching_pieces(x0, x1, t):
    x_t = ___            # 🧩 a point on the straight line from noise to the real chunk
    target = ___                        # 🧩 the constant velocity along that line
    return x_t, target

flow_matching_pieces = check("fm_pieces", flow_matching_pieces)

<details><summary>🤔 <b>Need a hint?</b></summary>

Exactly lab 07, now with chunks of shape (B, 16, 2).

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>x_t = (1 - t) * x0 + t * x1            # 🧩 a point on the straight line from noise to the real chunk
target = x1 - x0                        # 🧩 the constant velocity along that line</pre>

</details>

## 4 · Train ⏳
Two professional touches, both used by Diffusion Policy and π0:
* **EMA weights:** keep a slowly-moving average of the weights and use *it* at test time. It gives smoother, more reliable policies.
* **Gradient clipping + one-cycle learning rate** for stable training.

⏱ 20,000 steps took **~5 minutes on an Apple M-series laptop GPU** when we tested this notebook. A T4 should be in the same ballpark. The cell prints a live estimate.

In [ ]:
TRAIN_STEPS = 500 if FAST_DEV_RUN else 20000

def train_policy(loss_kind="flow", steps=TRAIN_STEPS, batch=256, lr=3e-4, seed=0, log_every=2000):
    torch.manual_seed(seed)
    net = FlowPolicy().to(DEVICE); ema = copy.deepcopy(net).eval()
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.05)
    losses, t0 = [], time.time()
    for step in range(steps):
        idx = torch.randint(0, len(OBS), (batch,), device=DEVICE)
        obs, x1 = OBS[idx], CHUNKS[idx]
        if loss_kind == "flow":
            x0 = torch.randn_like(x1)                              # noise
            t = torch.rand(batch, 1, 1, device=DEVICE)             # random flow time per example
            x_t, target = flow_matching_pieces(x0, x1, t)
            loss = F.mse_loss(net(x_t, t.view(batch, 1), obs), target)
        else:                                                      # regression baseline: predict the chunk directly
            zeros = torch.zeros_like(x1)
            loss = F.mse_loss(net(zeros, torch.zeros(batch, 1, device=DEVICE), obs), x1)
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sched.step()
        with torch.no_grad():                                      # EMA: ema = 0.999·ema + 0.001·net
            decay = min(0.999, (1 + step) / (10 + step))
            for pe, pn in zip(ema.parameters(), net.parameters()):
                pe.mul_(decay).add_(pn.detach(), alpha=1 - decay)
        losses.append(loss.item())
        if step % log_every == 0 or step == steps - 1:
            rate = (step + 1) / (time.time() - t0)
            print(f"step {step:>6} · loss {np.mean(losses[-200:]):.4f} · {rate:.0f} steps/s · ~{(steps - step) / rate / 60:.1f} min left")
    return ema, losses

policy, losses = train_policy("flow")
torch.save(policy.state_dict(), OUT / "c1_flow_policy.pt")
plt.figure(figsize=(5, 2.5)); plt.plot(np.convolve(losses, np.ones(100) / 100, "valid")); plt.yscale("log")
plt.xlabel("step"); plt.ylabel("loss"); plt.title("flow matching loss"); plt.show()

---
## 5 · From noise to a plan ✨
### 🧩 Challenge 5 · Integrate noise into actions

In [ ]:
@torch.no_grad()
def euler_integrate(velocity_net, obs, x, flow_steps):
    dt = 1.0 / flow_steps
    for k in range(flow_steps):
        t = torch.full((len(x), 1), k * dt, device=x.device)
        x = ___         # 🧩 one small step along the predicted velocity
    return x

euler_integrate = check("euler", euler_integrate)

@torch.no_grad()
def sample_chunks(net, prev_state, state, n=1, flow_steps=10):
    obs = torch.tensor(np.concatenate([state_features(prev_state), state_features(state)]), device=DEVICE)[None].repeat(n, 1)
    x = torch.randn(n, H, 2, device=DEVICE)                       # start from pure noise
    return from_unit(euler_integrate(net, obs, x, flow_steps).clamp(-1, 1)).cpu().numpy()

<details><summary>🤔 <b>Need a hint?</b></summary>

New x = x + dt × velocity.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>x = x + dt * velocity_net(x, t, obs)         # 🧩 one small step along the predicted velocity</pre>

</details>

### Many possible plans from one scene
Sample 30 chunks for the same moment. A generative policy can propose **different strategies**. Look for plans that go around different sides of the T.

In [ ]:
e = data["test_eps"][2]; i = starts[e] + 5
env = make_env("state"); env.reset(seed=0); set_dataset_state(env, S[i]); img = env.unwrapped.render(); k = img.shape[0] / 512
plans = sample_chunks(policy, S[i - 1], S[i], n=30)
plt.figure(figsize=(3.8, 3.8)); plt.imshow(img)
for p in plans:
    plt.plot(p[:, 0] * k, p[:, 1] * k, "-", lw=1, alpha=0.7)
plt.title("30 sampled 1.6-second plans"); plt.axis("off"); plt.show()

---
## 6 · Close the loop in the simulator 🔁
### 🧩 Challenge 6 · Receding horizon

In [ ]:
def actions_to_execute(chunk, execute):
    return ___                 # 🧩 carry out only the first part of the plan

actions_to_execute = check("execute", actions_to_execute)

def flow_controller(net, flow_steps=10, execute=EXECUTE):
    def choose(history, env):
        prev, now = history
        return actions_to_execute(sample_chunks(net, prev, now, flow_steps=flow_steps)[0], execute)
    return choose

seeds = EVAL_SEEDS[:5] if FAST_DEV_RUN else EVAL_SEEDS
flow_result = evaluate(flow_controller(policy), seeds=seeds, label="flow-matching policy")
print(f"human reference on the test demos: {HUMAN:.3f}")

<details><summary>🤔 <b>Need a hint?</b></summary>

Slice the chunk.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return chunk[:execute]                 # 🧩 carry out only the first part of the plan</pre>

</details>

### 🎬 Watch it: best and worst of the first 12 test scenes

In [ ]:
scores = flow_result["per_episode_scores"][:12]
best_seed, worst_seed = EVAL_SEEDS[int(np.argmax(scores))], EVAL_SEEDS[int(np.argmin(scores))]
for tag, seed in [("best", best_seed), ("worst", worst_seed)]:
    ep = run_episode(flow_controller(policy), seed, record=True)
    path = save_gif(ep["frames"], f"c1_{tag}.gif")
    print(f"{tag}: seed {seed} · score {ep['score']:.2f}"); show_gif(path)

---
## 7 · Experiments: what actually matters? 🔬

### Experiment A · Why a *generative* policy?
We train the **same network** as a plain regression policy (MSE on the chunk) with the same data, steps and evaluation.

In [ ]:
quiz("mse")

In [ ]:
mse_policy, mse_losses = train_policy("mse")
def mse_controller(net, execute=EXECUTE):
    @torch.no_grad()
    def choose(history, env):
        prev, now = history
        obs = torch.tensor(np.concatenate([state_features(prev), state_features(now)]), device=DEVICE)[None]
        chunk = net(torch.zeros(1, H, 2, device=DEVICE), torch.zeros(1, 1, device=DEVICE), obs).clamp(-1, 1)
        return from_unit(chunk)[0].cpu().numpy()[:execute]
    return choose
mse_result = evaluate(mse_controller(mse_policy), seeds=seeds, label="regression (MSE) policy")
print(f"final training loss · flow {np.mean(losses[-500:]):.4f} vs regression {np.mean(mse_losses[-500:]):.4f} (different objectives, so not comparable)")
diff = np.array(flow_result["per_episode_scores"]) - np.array(mse_result["per_episode_scores"])
m, lo, hi = bootstrap_ci(diff)
print(f"flow − regression, paired over the same scenes: {m:+.3f} (95% CI {lo:+.3f} to {hi:+.3f})")
print(f"scenes where a policy collapsed (score < 0.3): flow {sum(x < 0.3 for x in flow_result['per_episode_scores'])} · regression {sum(x < 0.3 for x in mse_result['per_episode_scores'])}")

### Experiment B · Flow steps and execution horizon
Each setting is evaluated on the first 20 test scenes. Same trained policy, only *inference* changes.

In [ ]:
quiz("steps")

In [ ]:
grid_seeds = EVAL_SEEDS[:3] if FAST_DEV_RUN else EVAL_SEEDS[:20]
ablation = {}
for fs in [1, 2, 5, 10]:
    for ex in [2, 8, 16]:
        r = evaluate(flow_controller(policy, flow_steps=fs, execute=ex), seeds=grid_seeds, execute=ex, verbose=False)
        ablation[(fs, ex)] = r["score"]
        print(f"flow steps {fs:>2} · execute {ex:>2} → score {r['score']:.3f}")
fig, ax = plt.subplots(figsize=(4.8, 3.2))
grid = np.array([[ablation[(fs, ex)] for ex in [2, 8, 16]] for fs in [1, 2, 5, 10]])
im = ax.imshow(grid, cmap="viridis"); plt.colorbar(im, label="score")
ax.set_xticks(range(3), [2, 8, 16]); ax.set_yticks(range(4), [1, 2, 5, 10]); ax.set_xlabel("actions executed per plan"); ax.set_ylabel("flow steps")
for (r_, c_), v in np.ndenumerate(grid): ax.text(c_, r_, f"{v:.2f}", ha="center", va="center", color="w", fontsize=8)
plt.title("inference-time choices"); plt.show()

---
## 8 · Save your results and write them up 📝

In [ ]:
summary = dict(stage="C1", human_reference=HUMAN, flow=flow_result, regression=mse_result,
               ablation={f"steps={k[0]},execute={k[1]}": v for k, v in ablation.items()}, train_steps=TRAIN_STEPS, device=str(DEVICE))
save_results("c1_results.json", summary)
lo, hi = flow_result["ci95"]
print("📄 Draft résumé bullet (edit to taste):")
print(f"  • Built a flow-matching action-chunk policy (6.4M params, π0-style action expert) for the PushT benchmark;"
      f" scored {flow_result['score']:.2f} (95% CI {lo:.2f}–{hi:.2f}) over 50 fixed simulator episodes vs {HUMAN:.2f} for human"
      f" demonstrations, beating an identically-sized regression policy ({mse_result['score']:.2f}).")
print("\nDownload the capstone_outputs/ folder (Colab: Files panel) and keep it for C3 and C4.")

### ✅ Stage checklist
- [ ] `capstone_outputs/c1_flow_policy.pt`, `c1_results.json`, `c1_best.gif`, `c1_worst.gif` downloaded
- [ ] One paragraph: *what failed in the worst GIF, and why might the policy do that?*
- [ ] One sentence on Experiment A that you'd say in an interview: *why does a generative policy beat regression on a multimodal task?*

### 🚀 Stretch ideas (pick one, optional)
* Sample `t` from a Beta distribution that emphasises noisy times (π0 does this). Does the score change?
* Replace the MLP with a small transformer over the 16 action tokens (closer to GR00T's DiT head).
* Train 3 seeds and report seed-to-seed spread separately from the episode interval (lab 03).

**Next:** `C2_Vision_Policy_PushT.ipynb`. The same policy, but it must *see* the block from pixels.

In [ ]:
progress_report()